<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/NAN_CHECK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GLM

In [ ]:
# ============================================================================
# INFERENCE (CORRECTED): reload the TOPO-2026 classifier and run predictions
# ============================================================================
# IMPORTANT: what was trained is a 3-task TEXT CLASSIFIER on top of a FROZEN
# GLM base. It cannot "chat" / generate free text -- that capability was never
# trained. So the correct test is classification, which is what this cell does.

import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import hf_hub_download

# ============================================================================
# DEFINE EVERYTHING
# ============================================================================

MODEL_NAME = "zai-org/GLM-4.6V-Flash"
REPO_ID = "frankmorales2020/glm-4.6-topo-2026"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TaskAwareGLM class - FIXED to match checkpoint format
class TaskAwareGLM(nn.Module):
    def __init__(self, base_model, hidden_size=4096, head_hidden=512):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size

        for param in self.base_model.parameters():
            param.requires_grad = False

        # Use single Linear layer (matches checkpoint format)
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)

        self.current_task = 'A'
        self._init_heads()

    def _init_heads(self):
        for head in [self.classifier_A, self.classifier_B, self.classifier_C]:
            nn.init.xavier_uniform_(head.weight)
            if head.bias is not None:
                nn.init.zeros_(head.bias)

    def switch_task(self, task):
        self.current_task = task

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False
        )
        hidden = outputs.hidden_states[-1]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)

        classifier = getattr(self, f'classifier_{self.current_task}')
        return classifier(pooled)

# get_embed_layer function
def get_embed_layer(model):
    if hasattr(model, 'embed_tokens'):
        return model.embed_tokens
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'embed_tokens'):
        return model.transformer.embed_tokens
    elif hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
        return model.model.embed_tokens
    elif hasattr(model, 'encoder') and hasattr(model.encoder, 'embed_tokens'):
        return model.encoder.embed_tokens
    elif hasattr(model, 'decoder') and hasattr(model.decoder, 'embed_tokens'):
        return model.decoder.embed_tokens
    else:
        for name, module in model.named_modules():
            if isinstance(module, nn.Embedding):
                return module
        raise ValueError("Could not find embedding layer in model")

# ============================================================================
# YOUR ORIGINAL CODE WITH FIXED MODEL CLASS
# ============================================================================

print("📥 Loading stock GLM base in 4-bit (same as training)...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16)
base_model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📥 Rebuilding TaskAwareGLM and loading trained parts...")
model = TaskAwareGLM(base_model).to(device)

ckpt = torch.load(hf_hub_download(REPO_ID, "topo_trained_parts.pt"),
                  map_location="cpu", weights_only=False)

# Load classifier heads directly (now they match)
model.classifier_A.load_state_dict(
    {k: v.to(model.classifier_A.weight.dtype)
     for k, v in ckpt["classifier_A"].items()}
)
model.classifier_B.load_state_dict(
    {k: v.to(model.classifier_B.weight.dtype)
     for k, v in ckpt["classifier_B"].items()}
)
model.classifier_C.load_state_dict(
    {k: v.to(model.classifier_C.weight.dtype)
     for k, v in ckpt["classifier_C"].items()}
)

# Restore trained embedding rows (copy into the live embedding table)
with torch.no_grad():
    emb = get_embed_layer(base_model)
    w = ckpt["embed_tokens_weight"].to(emb.weight.dtype).to(emb.weight.device)
    if w.shape == emb.weight.shape:
        emb.weight.copy_(w)
        print("  ✅ embedding weights restored")
    else:
        print(f"  ⚠️ embedding shape mismatch {tuple(w.shape)} vs {tuple(emb.weight.shape)}; skipped")
model.eval()
print("✅ Classifier ready. Anchors:", ckpt["prime_anchors"],
      "| Λ =", round(ckpt["safety_constant"], 6))

TASK_LABELS = {
    "A": ["World", "Sports"],
    "B": ["Business", "Science"],
    "C": ["Politics", "Technology"],
}

@torch.no_grad()
def classify(text, task):
    model.switch_task(task)
    tok = tokenizer([text], return_tensors="pt", padding=True,
                    truncation=True, max_length=ckpt.get("max_len", 64)).to(device)
    logits = model(tok.input_ids, tok.attention_mask)
    probs = torch.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs))
    return TASK_LABELS[task][idx], float(probs[idx])

print("\n" + "="*80 + "\n🚀 CLASSIFICATION TESTS\n" + "="*80)
tests = [
    ("World summit on climate policy",        "A"),
    ("The team won the championship match",   "A"),
    ("Quarterly business earnings report",    "B"),
    ("New scientific research breakthrough",  "B"),
    ("Government election and policy debate", "C"),
    ("Latest smartphone technology release",  "C"),
]
for text, task in tests:
    label, conf = classify(text, task)
    print(f"[{task}] {text!r:45s} -> {label:12s} ({conf*100:.1f}%)")

print("\n" + "="*80 + "\n🏁 INFERENCE COMPLETE\n" + "="*80)

# ============================================================================
# NAN DETECTION - ADDED AFTER INFERENCE
# ============================================================================
print("\n" + "="*80)
print("🔍 NAN DETECTION")
print("="*80)

results = {"has_nan": False, "has_inf": False}

# Check embedding layer
emb = get_embed_layer(base_model)
weight = emb.weight.float()

nan_count = torch.isnan(weight).sum().item()
inf_count = torch.isinf(weight).sum().item()

print(f"\n📊 Embedding Matrix ({weight.numel():,} elements):")
print(f"   Shape: {tuple(weight.shape)}")
print(f"   NaN: {nan_count:,}")
print(f"   Inf: {inf_count:,}")
print(f"   Mean: {weight.mean().item():.6f}")
print(f"   Std: {weight.std().item():.6f}")
print(f"   Min: {weight.min().item():.6f}")
print(f"   Max: {weight.max().item():.6f}")

if nan_count > 0:
    print(f"   ❌ CONTAINS {nan_count:,} NaN VALUES!")
    results["has_nan"] = True
else:
    print(f"   ✅ No NaN values")

if inf_count > 0:
    print(f"   ❌ CONTAINS {inf_count:,} Inf VALUES!")
    results["has_inf"] = True
else:
    print(f"   ✅ No Inf values")

# Check prime anchor rows
anchors = ckpt["prime_anchors"]
print(f"\n🎯 Prime Anchors: {anchors}")
anchor_nan = 0
anchor_inf = 0

for idx in anchors:
    if idx < weight.shape[0]:
        row = weight[idx]
        a_nan = torch.isnan(row).sum().item()
        a_inf = torch.isinf(row).sum().item()
        anchor_nan += a_nan
        anchor_inf += a_inf
        status = "✅" if a_nan == 0 and a_inf == 0 else "❌"
        print(f"   {status} Anchor {idx}: NaN={a_nan}, Inf={a_inf}")

if anchor_nan > 0:
    results["has_nan"] = True
    print(f"   ❌ Anchor rows contain {anchor_nan} NaN values")
else:
    print(f"   ✅ No NaN in anchor rows")

# Check classifier heads
print(f"\n📊 Classifier Heads:")
for head_name in ['classifier_A', 'classifier_B', 'classifier_C']:
    head = getattr(model, head_name)
    head_nan = 0
    head_inf = 0
    total_params = 0

    for param in head.parameters():
        p_f32 = param.float()
        head_nan += torch.isnan(p_f32).sum().item()
        head_inf += torch.isinf(p_f32).sum().item()
        total_params += p_f32.numel()

    status = "✅" if head_nan == 0 and head_inf == 0 else "❌"
    print(f"   {status} {head_name}: params={total_params:,}, NaN={head_nan}, Inf={head_inf}")

    if head_nan > 0 or head_inf > 0:
        results["has_nan"] = True
        results["has_inf"] = True

# Final summary
status = "PASS" if not results["has_nan"] and not results["has_inf"] else "FAIL"
print("\n" + "="*80)
print("📋 NAN DETECTION SUMMARY")
print("="*80)
print(f"   Status: {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
print(f"   Embedding NaN: {'❌ YES' if results['has_nan'] else '✅ NO'}")
print(f"   Embedding Inf: {'❌ YES' if results['has_inf'] else '✅ NO'}")
print("="*80)

📥 Loading stock GLM base in 4-bit (same as training)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/703 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Glm4vModel LOAD REPORT from: zai-org/GLM-4.6V-Flash
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📥 Rebuilding TaskAwareGLM and loading trained parts...
  ⚠️ embedding shape mismatch (151552, 4096) vs (576, 1536); skipped
✅ Classifier ready. Anchors: [2, 3, 5, 7, 11, 13] | Λ = 0.978514

🚀 CLASSIFICATION TESTS
[A] 'World summit on climate policy'              -> World        (100.0%)
[A] 'The team won the championship match'         -> Sports       (100.0%)
[B] 'Quarterly business earnings report'          -> Business     (99.6%)
[B] 'New scientific research breakthrough'        -> Science      (100.0%)
[C] 'Government election and policy debate'       -> Politics     (100.0%)
[C] 'Latest smartphone technology release'        -> Technology   (100.0%)

🏁 INFERENCE COMPLETE

🔍 NAN DETECTION

📊 Embedding Matrix (884,736 elements):
   Shape: (576, 1536)
   NaN: 0
   Inf: 0
   Mean: -0.001369
   Std: 0.662080
   Min: -45.750000
   Max: 44.500000
   ✅ No NaN values
   ✅ No Inf values

🎯 Prime Anchors: [2, 3, 5, 7, 11, 13]
   ✅ Anchor 2: NaN=0, Inf=0
   ✅ Anchor 3: NaN=0, Inf=0
   ✅ Anchor

In [ ]:
MODEL_NAME    = "zai-org/GLM-4.6V-Flash"

In [ ]:
!pip install -U bitsandbytes>=0.46.1B -q

## sarvam

In [ ]:
!pip install compressed-tensors>=0.15.0 -q

In [ ]:
# ============================================================================
# Sarvam-30B - NaN Detection (Fixed - word_embeddings)
# ============================================================================
# Based on model card: frankmorales2020/topological-ai-sarvam-30b-multirun
# Fix: Sarvam uses word_embeddings instead of embed_tokens

import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download
import gc
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("🔍 Sarvam-30B NaN Detection")
print("="*70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

# ============================================================================
# CONFIGURATION
# ============================================================================
BASE_MODEL = "sarvamai/sarvam-30b-fp8"
REPO_ID = "frankmorales2020/topological-ai-sarvam-30b-multirun"
HIDDEN_SIZE = 4096
HEAD_HIDDEN = 512

print(f"\n📋 Configuration:")
print(f"   BASE_MODEL: {BASE_MODEL}")
print(f"   REPO_ID: {REPO_ID}")
print(f"   HIDDEN_SIZE: {HIDDEN_SIZE}")

# ============================================================================
# DEFINE TASK-AWARE MODEL
# ============================================================================
class TaskAwareSarvam(nn.Module):
    def __init__(self, base_model, hidden_size=4096, head_hidden=512):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size

        for param in self.base_model.parameters():
            param.requires_grad = False

        def _head():
            return nn.Sequential(
                nn.Linear(hidden_size, head_hidden, dtype=torch.bfloat16),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(head_hidden, 2, dtype=torch.bfloat16)
            )

        self.classifier_A = _head()
        self.classifier_B = _head()
        self.classifier_C = _head()
        self.current_task = 'A'
        self._init_heads()

    def _init_heads(self):
        for head in [self.classifier_A, self.classifier_B, self.classifier_C]:
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def switch_task(self, task):
        self.current_task = task

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False
        )
        hidden = outputs.hidden_states[-1]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)

        classifier = getattr(self, f'classifier_{self.current_task}')
        return classifier(pooled)

# ============================================================================
# get_embedding_layer - Sarvam specific
# ============================================================================
def get_embedding_layer(model):
    """Get the embedding layer from Sarvam model."""
    # Sarvam uses word_embeddings
    if hasattr(model, 'word_embeddings'):
        return model.word_embeddings
    elif hasattr(model, 'embed_tokens'):
        return model.embed_tokens
    elif hasattr(model, 'model') and hasattr(model.model, 'word_embeddings'):
        return model.model.word_embeddings
    elif hasattr(model, 'model') and hasattr(model.model, 'embed_tokens'):
        return model.model.embed_tokens
    else:
        # Search for any Embedding layer
        for name, module in model.named_modules():
            if isinstance(module, nn.Embedding):
                print(f"   Found embedding layer: {name}")
                return module
        raise ValueError("Could not find embedding layer in model")

# ============================================================================
# LOAD BASE MODEL
# ============================================================================
print("\n📥 Loading Sarvam-30B base model...")

try:
    base_model = AutoModel.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True,
        dtype=torch.bfloat16,
        device_map=None,
        low_cpu_mem_usage=False,
    )
    base_model = base_model.cpu()

    # Convert FP8 tensors to bfloat16
    converted = 0
    for name, param in base_model.named_parameters():
        if param.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
            param.data = param.data.to(torch.bfloat16)
            converted += 1
    for name, buffer in base_model.named_buffers():
        if buffer.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
            buffer.data = buffer.data.to(torch.bfloat16)
            converted += 1
    if converted:
        print(f"   Converted {converted} FP8 tensors to bfloat16")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("✅ Base model loaded")

except Exception as e:
    print(f"❌ Failed to load base model: {e}")
    raise

# ============================================================================
# LOAD TRAINED PARTS
# ============================================================================
print("\n📥 Loading trained parts from checkpoint...")
model = TaskAwareSarvam(base_model, hidden_size=HIDDEN_SIZE).to(device)

try:
    ckpt_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="certified_topological_best.pt",
        local_files_only=False
    )
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    print(f"   Checkpoint keys: {list(checkpoint.keys())[:10]}")

    # Filter classifier keys
    filtered = {k: v for k, v in checkpoint.items() if 'classifier' in k}
    missing, unexpected = model.load_state_dict(filtered, strict=False)

    if missing:
        classifier_missing = [k for k in missing if 'classifier' in k]
        if classifier_missing:
            print(f"   ⚠️ Missing classifier keys: {classifier_missing[:5]}")
        else:
            print(f"   ✅ No missing classifier keys")

    print(f"✅ Loaded {len(filtered)} classifier tensors")

    # Restore embedding rows if available
    if 'word_embeddings.weight' in checkpoint:
        with torch.no_grad():
            emb = get_embedding_layer(base_model)
            w = checkpoint['word_embeddings.weight'].to(emb.weight.dtype).to(emb.weight.device)
            if w.shape == emb.weight.shape:
                emb.weight.copy_(w)
                print("  ✅ embedding weights restored")
            else:
                print(f"  ⚠️ embedding shape mismatch {tuple(w.shape)} vs {tuple(emb.weight.shape)}; skipped")
    elif 'embed_tokens_weight' in checkpoint:
        with torch.no_grad():
            emb = get_embedding_layer(base_model)
            w = checkpoint['embed_tokens_weight'].to(emb.weight.dtype).to(emb.weight.device)
            if w.shape == emb.weight.shape:
                emb.weight.copy_(w)
                print("  ✅ embedding weights restored")
            else:
                print(f"  ⚠️ embedding shape mismatch {tuple(w.shape)} vs {tuple(emb.weight.shape)}; skipped")
    else:
        print("  ⚠️ No embedding weights in checkpoint")

    anchors = checkpoint.get('prime_anchors', [2,3,5,7,11,13])
    safety = checkpoint.get('safety_constant', 0.9785142874)
    print(f"✅ Anchors: {anchors} | Λ = {round(safety, 6)}")

except Exception as e:
    print(f"❌ Failed to load checkpoint: {e}")
    import traceback
    traceback.print_exc()
    raise

model.eval()

# ============================================================================
# NAN DETECTION
# ============================================================================
print("\n" + "="*70)
print("🔍 NAN DETECTION")
print("="*70)

results = {"has_nan": False, "has_inf": False}

# Check embedding layer - FIXED: use get_embedding_layer
emb = get_embedding_layer(base_model)
weight = emb.weight.float()

nan_count = torch.isnan(weight).sum().item()
inf_count = torch.isinf(weight).sum().item()

print(f"\n📊 Embedding Matrix ({weight.numel():,} elements):")
print(f"   Shape: {tuple(weight.shape)}")
print(f"   Dtype: {emb.weight.dtype}")
print(f"   NaN: {nan_count:,}")
print(f"   Inf: {inf_count:,}")
print(f"   Mean: {weight.mean().item():.6f}")
print(f"   Std: {weight.std().item():.6f}")
print(f"   Min: {weight.min().item():.6f}")
print(f"   Max: {weight.max().item():.6f}")

if nan_count > 0:
    print(f"   ❌ CONTAINS {nan_count:,} NaN VALUES!")
    results["has_nan"] = True
else:
    print(f"   ✅ No NaN values")

if inf_count > 0:
    print(f"   ❌ CONTAINS {inf_count:,} Inf VALUES!")
    results["has_inf"] = True
else:
    print(f"   ✅ No Inf values")

# Check prime anchor rows
anchors = checkpoint.get('prime_anchors', [2,3,5,7,11,13])
print(f"\n🎯 Prime Anchors: {anchors}")
anchor_nan = 0
anchor_inf = 0

for idx in anchors:
    if idx < weight.shape[0]:
        row = weight[idx]
        a_nan = torch.isnan(row).sum().item()
        a_inf = torch.isinf(row).sum().item()
        anchor_nan += a_nan
        anchor_inf += a_inf
        status = "✅" if a_nan == 0 and a_inf == 0 else "❌"
        print(f"   {status} Anchor {idx}: NaN={a_nan}, Inf={a_inf}")

if anchor_nan > 0:
    results["has_nan"] = True
    print(f"   ❌ Anchor rows contain {anchor_nan} NaN values")
else:
    print(f"   ✅ No NaN in anchor rows")

# Check classifier heads
print(f"\n📊 Classifier Heads:")
for head_name in ['classifier_A', 'classifier_B', 'classifier_C']:
    head = getattr(model, head_name)
    head_nan = 0
    head_inf = 0
    total_params = 0

    for param in head.parameters():
        p_f32 = param.float()
        head_nan += torch.isnan(p_f32).sum().item()
        head_inf += torch.isinf(p_f32).sum().item()
        total_params += p_f32.numel()

    status = "✅" if head_nan == 0 and head_inf == 0 else "❌"
    print(f"   {status} {head_name}: params={total_params:,}, NaN={head_nan}, Inf={head_inf}")

    if head_nan > 0 or head_inf > 0:
        results["has_nan"] = True
        results["has_inf"] = True

# Final summary
status = "PASS" if not results["has_nan"] and not results["has_inf"] else "FAIL"
print("\n" + "="*80)
print("📋 NAN DETECTION SUMMARY")
print("="*80)
print(f"   Status: {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
print(f"   Embedding NaN: {'❌ YES' if results['has_nan'] else '✅ NO'}")
print(f"   Embedding Inf: {'❌ YES' if results['has_inf'] else '✅ NO'}")
print("="*80)

# Clean up
del base_model, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\n✅ Sarvam-30B check complete")

🔍 Sarvam-30B NaN Detection
🖥️  Device: cuda

📋 Configuration:
   BASE_MODEL: sarvamai/sarvam-30b-fp8
   REPO_ID: frankmorales2020/topological-ai-sarvam-30b-multirun
   HIDDEN_SIZE: 4096

📥 Loading Sarvam-30B base model...


[transformers] Unknown quantization type, got modelopt - supported types are: ['awq', 'bitsandbytes_4bit', 'bitsandbytes_8bit', 'gptq', 'aqlm', 'quanto', 'quark', 'fouroversix', 'fp_quant', 'eetq', 'higgs', 'hqq', 'compressed-tensors', 'fbgemm_fp8', 'torchao', 'bitnet', 'vptq', 'spqr', 'fp8', 'auto-round', 'mxfp4', 'metal', 'sinq']. Hence, we will skip the quantization. To remove the warning, you can delete the quantization_config attribute in config.json


Loading weights:   0%|          | 0/7121 [00:00<?, ?it/s]

[transformers] SarvamMoEModel LOAD REPORT from: sarvamai/sarvam-30b-fp8
Key                                                                | Status     |  | 
-------------------------------------------------------------------+------------+--+-
model.layers.{1...18}.mlp.experts.{0...127}.gate_proj.input_scale  | UNEXPECTED |  | 
model.layers.{1...18}.mlp.experts.{0...127}.up_proj.input_scale    | UNEXPECTED |  | 
model.layers.{1...18}.mlp.experts.{0...127}.down_proj.weight_scale | UNEXPECTED |  | 
model.layers.{1...18}.mlp.experts.{0...127}.up_proj.weight_scale   | UNEXPECTED |  | 
model.layers.{1...18}.mlp.experts.{0...127}.down_proj.input_scale  | UNEXPECTED |  | 
model.layers.{1...18}.mlp.experts.{0...127}.gate_proj.weight_scale | UNEXPECTED |  | 
model.layers.{0...18}.attention.dense.input_scale                  | UNEXPECTED |  | 
model.layers.{0...18}.attention.query_key_value.weight_scale       | UNEXPECTED |  | 
model.layers.{1...18}.mlp.shared_experts.down_proj.weight_scale    |

✅ Base model loaded

📥 Loading trained parts from checkpoint...
   Checkpoint keys: ['base_model.model.word_embeddings.weight', 'base_model.model.layers.0.attention.query_key_value.weight_scale', 'base_model.model.layers.0.attention.query_key_value.weight', 'base_model.model.layers.0.attention.query_layernorm.weight', 'base_model.model.layers.0.attention.key_layernorm.weight', 'base_model.model.layers.0.attention.dense.weight_scale', 'base_model.model.layers.0.attention.dense.weight', 'base_model.model.layers.0.mlp.gate_proj.weight_scale', 'base_model.model.layers.0.mlp.gate_proj.weight', 'base_model.model.layers.0.mlp.up_proj.weight_scale']
   ✅ No missing classifier keys
✅ Loaded 12 classifier tensors
  ⚠️ No embedding weights in checkpoint
✅ Anchors: [2, 3, 5, 7, 11, 13] | Λ = 0.978514

🔍 NAN DETECTION

📊 Embedding Matrix (1,073,741,824 elements):
   Shape: (262144, 4096)
   Dtype: torch.bfloat16
   NaN: 0
   Inf: 0
   Mean: -0.000046
   Std: 0.087285
   Min: -8.750000
   Max: 2.234

## DS

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.38.0 -q
!pip install accelerate bitsandbytes -q

In [ ]:
!pip install -q "compressed-tensors>=0.15.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 167.4 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# DeepSeek-V2-Lite - NaN Detection (Using Your Working Code)
# ============================================================================
# Run once, then RESTART RUNTIME, then run this cell:
#   !pip install -q "compressed-tensors>=0.15.0"

import warnings, logging, os
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
try:
    from transformers.utils import logging as _hf_log
    _hf_log.set_verbosity_error()
except Exception:
    pass

import transformers.utils.import_utils as _iu
import transformers.utils as _tu
for _m in (_iu, _tu):
    if not hasattr(_m, "is_torch_fx_available"):
        _m.is_torch_fx_available = lambda: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers.modeling_utils as _mu

# FP8 fix: skip re-init of already-loaded FP8 weights.
_mu.PreTrainedModel._initialize_weights = lambda self, *a, **k: None
_mu.PreTrainedModel.initialize_weights  = lambda self, *a, **k: None

print("="*70)
print("🔍 DeepSeek-V2-Lite NaN Detection")
print("="*70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

# ============================================================================
# CONFIGURATION - Using your repo
# ============================================================================
REPO = "frankmorales2020/deepseek-v2-lite-fp8-topo2026"
#REPO = "frankmorales2020/deepseek-governed-no-amnesia"

print(f"\n📋 Repository: {REPO}")

# ============================================================================
# LOAD MODEL (YOUR WORKING CODE)
# ============================================================================
print("\n📥 Loading DeepSeek-V2-Lite base model...")

try:
    tok = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
except Exception:
    tok = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V2-Lite", trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    REPO, dtype=torch.float16, device_map="auto",
    trust_remote_code=True, attn_implementation="eager").eval()

print("✅ Model loaded successfully")

# ============================================================================
# NAN DETECTION - YOUR CODE CHECK
# ============================================================================
print("\n" + "="*70)
print("🔍 NAN DETECTION")
print("="*70)

results = {"has_nan": False, "has_inf": False}

# Check embedding layer
emb = model.get_input_embeddings()
weight = emb.weight.float()

nan_count = torch.isnan(weight).sum().item()
inf_count = torch.isinf(weight).sum().item()

print(f"\n📊 Embedding Matrix ({weight.numel():,} elements):")
print(f"   Shape: {tuple(weight.shape)}")
print(f"   Dtype: {emb.weight.dtype}")
print(f"   NaN: {nan_count:,}")
print(f"   Inf: {inf_count:,}")
print(f"   Mean: {weight.mean().item():.6f}")
print(f"   Std: {weight.std().item():.6f}")
print(f"   Min: {weight.min().item():.6f}")
print(f"   Max: {weight.max().item():.6f}")

if nan_count > 0:
    print(f"   ❌ CONTAINS {nan_count:,} NaN VALUES!")
    results["has_nan"] = True
else:
    print(f"   ✅ No NaN values")

if inf_count > 0:
    print(f"   ❌ CONTAINS {inf_count:,} Inf VALUES!")
    results["has_inf"] = True
else:
    print(f"   ✅ No Inf values")

# Check prime anchor rows (if available in model)
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
print(f"\n🎯 Prime Anchors: {PRIME_ANCHORS}")
anchor_nan = 0
anchor_inf = 0

for idx in PRIME_ANCHORS:
    if idx < weight.shape[0]:
        row = weight[idx]
        a_nan = torch.isnan(row).sum().item()
        a_inf = torch.isinf(row).sum().item()
        anchor_nan += a_nan
        anchor_inf += a_inf
        status = "✅" if a_nan == 0 and a_inf == 0 else "❌"
        print(f"   {status} Anchor {idx}: NaN={a_nan}, Inf={a_inf}")

if anchor_nan > 0:
    results["has_nan"] = True
    print(f"   ❌ Anchor rows contain {anchor_nan} NaN values")
else:
    print(f"   ✅ No NaN in anchor rows")

# Check all model parameters for NaN
print(f"\n📊 Scanning all model parameters for NaN...")
param_nan = 0
param_inf = 0
total_params = 0

for name, param in model.named_parameters():
    p_f32 = param.float()
    p_nan = torch.isnan(p_f32).sum().item()
    p_inf = torch.isinf(p_f32).sum().item()
    param_nan += p_nan
    param_inf += p_inf
    total_params += p_f32.numel()
    if p_nan > 0 or p_inf > 0:
        print(f"   ⚠️  {name}: NaN={p_nan}, Inf={p_inf}")

if param_nan > 0:
    results["has_nan"] = True
    print(f"   ❌ Model contains {param_nan:,} NaN values across {total_params:,} parameters")
else:
    print(f"   ✅ No NaN values across {total_params:,} parameters")

if param_inf > 0:
    results["has_inf"] = True
    print(f"   ❌ Model contains {param_inf:,} Inf values")
else:
    print(f"   ✅ No Inf values")

# ============================================================================
# INFERENCE TEST (YOUR WORKING CODE)
# ============================================================================
print("\n" + "="*70)
print("🚀 INFERENCE TEST")
print("="*70)

PROMPTS = ["What is the capital of Japan?",
           "What is the Spanish word for water?",
           "What is Einstein's mass-energy equivalence?"]

print("\n📝 Running inference tests:")
for p in PROMPTS:
    try:
        ins = tok(p, return_tensors="pt").to(model.device)
        out = model.generate(**ins, max_new_tokens=40, do_sample=False,
                             repetition_penalty=1.2, no_repeat_ngram_size=3,
                             use_cache=False,
                             pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0], skip_special_tokens=True)
        print(f"\nQ: {p}")
        print(f"A: {txt[len(p):].strip() if txt.startswith(p) else txt.strip()}")
    except Exception as e:
        print(f"\nQ: {p}")
        print(f"A: ERROR - {e}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("📋 NAN DETECTION SUMMARY")
print("="*80)

status = "PASS" if not results["has_nan"] and not results["has_inf"] else "FAIL"
print(f"   Status: {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
print(f"   Embedding NaN: {'❌ YES' if results['has_nan'] else '✅ NO'}")
print(f"   Embedding Inf: {'❌ YES' if results['has_inf'] else '✅ NO'}")
print(f"   Model NaN: {'❌ YES' if param_nan > 0 else '✅ NO'}")
print(f"   Model Inf: {'❌ YES' if param_inf > 0 else '✅ NO'}")
print("="*80)
print("\n✅ DeepSeek-V2-Lite check complete")

🔍 DeepSeek-V2-Lite NaN Detection
🖥️  Device: cuda

📋 Repository: frankmorales2020/deepseek-v2-lite-fp8-topo2026

📥 Loading DeepSeek-V2-Lite base model...


config.json:   0%|          | 0.00/88.1k [00:00<?, ?B/s]

configuration_deepseek.py:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.50M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/12217 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded successfully

🔍 NAN DETECTION

📊 Embedding Matrix (209,715,200 elements):
   Shape: (102400, 2048)
   Dtype: torch.float16
   NaN: 0
   Inf: 0
   Mean: -0.000516
   Std: 0.131528
   Min: -2.046875
   Max: 1.664062
   ✅ No NaN values
   ✅ No Inf values

🎯 Prime Anchors: [2, 3, 5, 7, 11, 13]
   ✅ Anchor 2: NaN=0, Inf=0
   ✅ Anchor 3: NaN=0, Inf=0
   ✅ Anchor 5: NaN=0, Inf=0
   ✅ Anchor 7: NaN=0, Inf=0
   ✅ Anchor 11: NaN=0, Inf=0
   ✅ Anchor 13: NaN=0, Inf=0
   ✅ No NaN in anchor rows

📊 Scanning all model parameters for NaN...
   ✅ No NaN values across 15,706,491,150 parameters
   ✅ No Inf values

🚀 INFERENCE TEST

📝 Running inference tests:


Decompressing model: 100%|██████████| 3463/3463 [00:00<00:00, 12534.66it/s]



Q: What is the capital of Japan?
A: The answer to this question may seem obvious, but it’s actually a bit more complicated than you might think. The country has two capitals: Tokyo and Kyoto. Both cities are important in Japanese

Q: What is the Spanish word for water?
A: The English translation of “agua” in a sentence. Agua means Water, and it’s pronounced like this: ah-gwah (like how you say ‘gwa’

Q: What is Einstein's mass-energy equivalence?
A: Einstein’s Mass Energy Equivalence states that the energy of a body at rest (E) equals its mass times the speed of light squared. This equation can be written as: E = mc

📋 NAN DETECTION SUMMARY
   Status: ✅ PASS
   Embedding NaN: ✅ NO
   Embedding Inf: ✅ NO
   Model NaN: ✅ NO
   Model Inf: ✅ NO

✅ DeepSeek-V2-Lite check complete


## MIXTRAL

In [ ]:
!pip install compressed-tensors>=0.15.0 -q

In [ ]:
!pip show compressed-tensors transformers torch

Name: compressed-tensors
Version: 0.17.1
Summary: Library for utilization of compressed safetensors of neural network models
Home-page: https://github.com/vllm-project/compressed-tensors
Author: The vLLM Project
Author-email: vllm-questions@lists.berkeley.edu
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: loguru, pydantic, torch, transformers
Required-by: 
---
Name: transformers
Version: 5.10.2
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, saf

In [ ]:
# ============================================================================
# REAL MULTI-AGENT SYSTEM WITH ACTUAL MIXTRAL MODEL LOADING - CORRECTED v2
# Head architecture, dtype, activation, bias-correction and fallback logic
# now match the reference notebook (MIXTRAL_TOPO_INFERENCE).
#
# Root cause of the earlier "heads NONE" failure:
#   The checkpoint heads are nn.Sequential (Linear->GELU->Dropout->Linear),
#   with keys classifier_X.0.* and classifier_X.3.*, but the heads were
#   defined as a single nn.Linear (classifier_X.weight). Keys didn't match,
#   so load_state_dict(strict=False) silently dropped all 12 tensors.
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import numpy as np
import json
import gc
import random
import time
from typing import Any
from dataclasses import dataclass
from enum import Enum

# ============================================================================
# SET SEED
# ============================================================================
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
print("🎲 Random seed set to 123")

# ============================================================================
# CONFIGURATION  (mirrors the reference notebook)
# ============================================================================
HF_REPO_ID    = 'frankmorales2020/topological-ai-mixtral-8x7b-multirun'
BASE_MODEL_ID = 'frankmorales2020/mixtral-8x7b-fp8-topo2026'
HIDDEN_SIZE   = 4096
HEAD_HIDDEN   = 512
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

TASK_CONFIG = {
    'A': {'name': 'World vs Sports',     'labels': {0: 'World', 1: 'Sports'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'B': {'name': 'Business vs Sci/Tech','labels': {0: 'Business', 1: 'Sci/Tech'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'C': {'name': 'World vs Sci/Tech',   'labels': {0: 'World', 1: 'Sci/Tech'},
          'confidence_threshold': 0.80, 'fallback_task': 'B', 'bias_correction': -0.5},
}

# ============================================================================
# DISABLE COMPRESSED TENSORS HOOKS
# ============================================================================
def disable_compressed_tensors_hooks(model):
    for module in model.modules():
        for hook_dict_name in ('_forward_pre_hooks', '_forward_hooks'):
            hook_dict = getattr(module, hook_dict_name, None)
            if hook_dict:
                to_remove = [hid for hid, hook in hook_dict.items()
                             if 'compress' in str(hook).lower() or 'decompress' in str(hook).lower()]
                for hid in to_remove:
                    del hook_dict[hid]
    return model

# ============================================================================
# LOAD REAL MODEL
# ============================================================================
print("=" * 70)
print("LOADING REAL MIXTRAL MODEL")
print("=" * 70)

print(f"📥 Loading base model from {BASE_MODEL_ID}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map=None,
    low_cpu_mem_usage=False,
    ignore_mismatched_sizes=True,
)
base_model = base_model.cpu()

print("🔧 Removing compression hooks...")
base_model = disable_compressed_tensors_hooks(base_model)

print("🔄 Converting FP8 tensors to bfloat16...")
converted = 0
for _, param in base_model.named_parameters():
    if param.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        param.data = param.data.to(torch.bfloat16); converted += 1
for _, buffer in base_model.named_buffers():
    if buffer.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        buffer.data = buffer.data.to(torch.bfloat16); converted += 1
print(f"   Converted {converted} FP8 tensors")

print("🧹 Removing compression attributes...")
attrs_removed = 0
for module in base_model.modules():
    for attr in ['input_scale', 'weight_scale', 'down_proj_scale', 'gate_up_proj_scale',
                 'scales', 'zero_points', 'quantization_scheme', 'compression_config']:
        if hasattr(module, attr):
            delattr(module, attr); attrs_removed += 1
print(f"   Removed {attrs_removed} compression attributes")

gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================================
# TASK-AWARE MODEL  (head arch matches the checkpoint: Sequential .0/.1/.2/.3)
# ============================================================================
class RealTaskAwareModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        for param in self.base_model.parameters():
            param.requires_grad = False

        def _head():
            return nn.Sequential(
                nn.Linear(HIDDEN_SIZE, HEAD_HIDDEN, dtype=torch.bfloat16),  # .0
                nn.GELU(),                                                  # .1
                nn.Dropout(0.1),                                            # .2
                nn.Linear(HEAD_HIDDEN, 2, dtype=torch.bfloat16),           # .3
            )

        self.classifier_A = _head()
        self.classifier_B = _head()
        self.classifier_C = _head()
        self.current_task = 'A'
        self.loaded_heads = set()   # populated after verified checkpoint load
        self._init_heads()

    def _init_heads(self):
        for head in (self.classifier_A, self.classifier_B, self.classifier_C):
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def switch_task(self, task):
        self.current_task = task

    def forward(self, input_ids, attention_mask=None):
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            hidden = outputs.hidden_states[-1]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)

        classifier = getattr(self, f'classifier_{self.current_task}')
        return classifier(pooled)

    @torch.no_grad()
    def predict_task(self, text, task='A'):
        """Single-head prediction with the task's bias correction applied."""
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        self.switch_task(task)
        logits = self.forward(inputs['input_ids'], inputs['attention_mask'])

        bias = TASK_CONFIG[task].get('bias_correction', 0.0)
        if bias != 0.0:
            logits[:, 0] += bias

        probs = F.softmax(logits.float(), dim=-1)
        pred_class = probs.argmax().item()
        confidence = probs.max().item()
        label = TASK_CONFIG[task]['labels'][pred_class]
        return label, confidence, pred_class

    @torch.no_grad()
    def predict_with_fallback(self, text, task='C'):
        """Primary prediction; fall back to fallback_task if below threshold."""
        cfg = TASK_CONFIG[task]
        label, conf, cls = self.predict_task(text, task)
        if conf >= cfg['confidence_threshold'] or not cfg.get('fallback_task'):
            return label, conf, 'primary'
        fb = cfg['fallback_task']
        fb_label, fb_conf, _ = self.predict_task(text, fb)
        return fb_label, fb_conf, f'fallback->{fb}'

# ============================================================================
# CREATE MODEL + VERIFIED CHECKPOINT LOAD
# ============================================================================
print("\n" + "=" * 70)
print("CREATING TASK-AWARE MODEL")
print("=" * 70)

real_model = RealTaskAwareModel(base_model)

def head_signature(model, name):
    seq = getattr(model, name)
    sig = []
    for layer in seq:
        if isinstance(layer, nn.Linear):
            sig.append(round(layer.weight.float().abs().sum().item(), 4))
            sig.append(round(layer.bias.float().abs().sum().item(), 4))
    return tuple(sig)

pre_load_sig = {n: head_signature(real_model, n)
                for n in ('classifier_A', 'classifier_B', 'classifier_C')}

print("📦 Attempting to load certified weights...")
try:
    checkpoint_path = hf_hub_download(repo_id=HF_REPO_ID, filename='certified_topological_best.pt')
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Match checkpoint dtype to the bfloat16 heads.
    for k, v in checkpoint.items():
        if isinstance(v, torch.Tensor) and v.dtype == torch.float32:
            checkpoint[k] = v.to(torch.bfloat16)

    filtered = {k: v for k, v in checkpoint.items() if k.startswith('classifier_')}
    missing, unexpected = real_model.load_state_dict(filtered, strict=False)
    print(f"✓ Filtered {len(filtered)} classifier tensors from checkpoint")

    for name in ('classifier_A', 'classifier_B', 'classifier_C'):
        if head_signature(real_model, name) != pre_load_sig[name]:
            real_model.loaded_heads.add(name[-1])

    trusted = sorted(real_model.loaded_heads)
    untrusted = [t for t in ('A', 'B', 'C') if t not in real_model.loaded_heads]
    print(f"  ✅ Trained heads loaded: {trusted if trusted else 'NONE'}")
    if untrusted:
        print(f"  ⚠️  Heads still at RANDOM init: {untrusted}")
    cls_unexpected = [k for k in unexpected if 'classifier' in k]
    if cls_unexpected:
        print(f"  ⚠️  Unexpected classifier keys (arch mismatch): {cls_unexpected[:6]}")
except Exception as e:
    print(f"⚠️ Could not load certified weights: {e}")
    print("   Using randomly initialized classifiers — outputs are NOT meaningful.")

del pre_load_sig
print(f"📱 Moving model to {DEVICE}...")
real_model = real_model.to(DEVICE)
real_model.eval()
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print(f"✓ Base dtype: {next(real_model.base_model.parameters()).dtype}")
print(f"✓ Head dtype: {next(real_model.classifier_A.parameters()).dtype}")

# ============================================================================
# AGENTS
# ============================================================================
class AgentRole(Enum):
    CLASSIFIER = "classifier"
    TOPIC = "topic"
    SENTIMENT = "sentiment"
    DECISION = "decision"

@dataclass
class AgentOutput:
    agent: str
    result: Any
    confidence: float

class RealClassifierAgent:
    """
    The three heads are binary classifiers on different axes, so voting across
    them is invalid (Task B can't express "Sports", etc.). Instead we ROUTE each
    document to the single head whose axis fits, using keyword signals, then
    trust only that head. Task C keeps its bias-correction + fallback-to-B.
    """
    # Keyword signals -> which binary head can actually decide this document.
    ROUTING = {
        'A': ['game', 'team', 'win', 'won', 'championship', 'league', 'match',
              'victory', 'series', 'cup', 'tournament', 'season', 'player',
              'coach', 'score', 'playoff'],
        'B': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding',
              'profit', 'shares', 'quarterly', 'investor', 'ipo', 'merger'],
    }

    def __init__(self, model):
        self.model = model

    # Health/science/general-news terms belong on the World axis; if these are
    # present we route to C and suppress the B fallback (B can't express World).
    WORLD_SIGNAL = ['cancer', 'treatment', 'medical', 'drug', 'disease', 'patient',
                    'vaccine', 'health', 'president', 'government', 'election',
                    'war', 'climate', 'court', 'nasa', 'space', 'mission']

    def _route(self, text):
        tl = text.lower()
        a_hits = sum(1 for kw in self.ROUTING['A'] if kw in tl)
        b_hits = sum(1 for kw in self.ROUTING['B'] if kw in tl)
        w_hits = sum(1 for kw in self.WORLD_SIGNAL if kw in tl)
        if a_hits and a_hits >= b_hits:
            return 'A'
        # Strong world/health/science signal outweighs a stray business keyword.
        if w_hits and w_hits >= b_hits:
            return 'C'
        if b_hits:
            return 'B'
        return 'C'

    def process(self, text):
        task = self._route(text)

        if task == 'C':
            tl = text.lower()
            has_world = any(kw in tl for kw in self.WORLD_SIGNAL)
            if has_world:
                # Trust C directly; B's fallback can't express World and would
                # mislabel general/health news as Business/Sci-Tech.
                label, conf, _ = self.model.predict_task(text, 'C')
                source = 'primary'
            else:
                label, conf, source = self.model.predict_with_fallback(text, 'C')
        else:
            label, conf, _ = self.model.predict_task(text, task)
            source = 'primary'

        return AgentOutput(
            agent='Classifier',
            result={
                'category': label,
                'routed_task': task,
                'task_c_source': source,
            },
            confidence=conf
        )

class RealTopicAgent:
    def process(self, text):
        topics = []
        text_lower = text.lower()
        topic_keywords = {
            'sports': ['game', 'team', 'win', 'championship', 'league', 'match', 'victory'],
            'business': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding', 'profit'],
            'technology': ['ai', 'quantum', 'software', 'cloud', 'algorithm', 'chip', 'computing'],
            'health': ['cancer', 'treatment', 'medical', 'drug', 'disease', 'breakthrough'],
            'politics': ['president', 'government', 'policy', 'election', 'law', 'congress'],
            'space': ['nasa', 'mission', 'space', 'moon', 'mars', 'jupiter', 'launch'],
        }
        for topic, keywords in topic_keywords.items():
            if any(kw in text_lower for kw in keywords):
                topics.append(topic)
        return AgentOutput(agent='Topic', result=topics if topics else ['general'],
                           confidence=0.8 if topics else 0.5)

class RealSentimentAgent:
    def process(self, text):
        text_lower = text.lower()
        positive = ['won', 'victory', 'breakthrough', 'growth', 'strong', 'record', 'beat', 'secured', 'surged']
        negative = ['loss', 'defeat', 'crisis', 'decline', 'risk', 'concern', 'failed']
        pos = sum(1 for w in positive if w in text_lower)
        neg = sum(1 for w in negative if w in text_lower)
        if pos > neg:
            return AgentOutput(agent='Sentiment', result='positive', confidence=0.8)
        elif neg > pos:
            return AgentOutput(agent='Sentiment', result='negative', confidence=0.8)
        return AgentOutput(agent='Sentiment', result='neutral', confidence=0.5)

class RealDecisionAgent:
    def process(self, classifier_out, topic_out, sentiment_out):
        category = classifier_out.result['category']
        confidence = classifier_out.confidence
        source = classifier_out.result.get('task_c_source', 'primary')
        is_fallback = source.startswith('fallback')
        # Auto-approve only confident, non-fallback predictions.
        if confidence >= 0.75 and not is_fallback:
            action, review = 'auto_approve', False
        else:
            action, review = 'review', True
        routing = {
            'Sports': 'sports_desk', 'Business': 'finance_team',
            'Sci/Tech': 'tech_research', 'World': 'news_department',
        }.get(category, 'general_inbox')
        return AgentOutput(
            agent='Decision',
            result={'category': category, 'action': action, 'routing': routing,
                    'requires_review': review, 'topics': topic_out.result,
                    'sentiment': sentiment_out.result},
            confidence=confidence
        )

# ============================================================================
# ORCHESTRATOR
# ============================================================================
class RealOrchestrator:
    def __init__(self, model):
        self.classifier = RealClassifierAgent(model)
        self.topic = RealTopicAgent()
        self.sentiment = RealSentimentAgent()
        self.decision = RealDecisionAgent()

    def analyze(self, text, verbose=True):
        start = time.time()
        classifier_out = self.classifier.process(text)
        topic_out = self.topic.process(text)
        sentiment_out = self.sentiment.process(text)
        decision_out = self.decision.process(classifier_out, topic_out, sentiment_out)
        elapsed = (time.time() - start) * 1000

        if verbose:
            c = classifier_out.result
            print(f"\n{'=' * 55}")
            print(f"📝 {text[:65]}...")
            print(f"{'=' * 55}")
            print(f"🎯 {decision_out.result['category']} ({classifier_out.confidence * 100:.0f}%)"
                  f"  [routed->Task {c['routed_task']}, source {c['task_c_source']}]")
            print(f"🏷️  {', '.join(topic_out.result[:3])}")
            print(f"💭 {sentiment_out.result}")
            print(f"✅ {decision_out.result['action']} → {decision_out.result['routing']}")
            print(f"⏱️  {elapsed:.0f}ms")

        return {
            'text': text[:100],
            'category': decision_out.result['category'],
            'confidence': classifier_out.confidence,
            'routed_task': classifier_out.result['routed_task'],
            'task_c_source': classifier_out.result['task_c_source'],
            'topics': topic_out.result,
            'sentiment': sentiment_out.result,
            'action': decision_out.result['action'],
            'routing': decision_out.result['routing'],
            'time_ms': elapsed,
        }

# ============================================================================
# RUN
# ============================================================================
print("\n" + "=" * 70)
print("RUNNING REAL MULTI-AGENT SYSTEM")
print("=" * 70)

orchestrator = RealOrchestrator(real_model)

test_docs = [
    "The Chicago Cubs won the World Series championship",
    "Microsoft stock surged 15% after beating earnings",
    "NASA launched a mission to explore Jupiter's moon Europa",
    "Scientists discovered breakthrough treatment for cancer",
    "Apple reported record iPhone sales and new AI features",
]

results = []
for doc in test_docs:
    results.append(orchestrator.analyze(doc, verbose=True))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
for i, r in enumerate(results, 1):
    icon = "✅" if r['action'] == 'auto_approve' else "⚠️"
    print(f"{icon} {i}. {r['category']:10s} | {r['action']:12s} | "
          f"{r['confidence'] * 100:.0f}% (Task {r['routed_task']})")

with open('real_model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n💾 Saved to 'real_model_results.json'")
print("\n✅ REAL MODEL SYSTEM READY")

🎲 Random seed set to 123
LOADING REAL MIXTRAL MODEL
📥 Loading base model from frankmorales2020/mixtral-8x7b-fp8-topo2026...


Compressing model: 100%|██████████| 128/128 [00:00<00:00, 1935.60it/s]


Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

[transformers] MixtralForCausalLM LOAD REPORT from: frankmorales2020/mixtral-8x7b-fp8-topo2026
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
model.layers.{0...31}.mlp.experts.gate_up_proj_scale                      | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w1.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w2.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w3.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.down_proj_scale                         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔧 Removing compression hooks...
🔄 Converting FP8 tensors to bfloat16...
   Converted 192 FP8 tensors
🧹 Removing compression attributes...
   Removed 384 compression attributes
📚 Loading tokenizer...

CREATING TASK-AWARE MODEL
📦 Attempting to load certified weights...
✓ Filtered 12 classifier tensors from checkpoint
  ✅ Trained heads loaded: ['A', 'B', 'C']
📱 Moving model to cuda...
✓ Base dtype: torch.bfloat16
✓ Head dtype: torch.bfloat16

RUNNING REAL MULTI-AGENT SYSTEM

📝 The Chicago Cubs won the World Series championship...
🎯 Sports (92%)  [routed->Task A, source primary]
🏷️  sports
💭 positive
✅ auto_approve → sports_desk
⏱️  1181ms

📝 Microsoft stock surged 15% after beating earnings...
🎯 Business (98%)  [routed->Task B, source primary]
🏷️  business
💭 positive
✅ auto_approve → finance_team
⏱️  67ms

📝 NASA launched a mission to explore Jupiter's moon Europa...
🎯 World (94%)  [routed->Task C, source primary]
🏷️  space
💭 neutral
✅ auto_approve → news_department
⏱️  67ms

📝 Scientists

In [ ]:
# ============================================================================
# REAL MULTI-AGENT SYSTEM WITH ACTUAL MIXTRAL MODEL LOADING - CORRECTED v2
# Head architecture, dtype, activation, bias-correction and fallback logic
# now match the reference notebook (MIXTRAL_TOPO_INFERENCE).
#
# Root cause of the earlier "heads NONE" failure:
#   The checkpoint heads are nn.Sequential (Linear->GELU->Dropout->Linear),
#   with keys classifier_X.0.* and classifier_X.3.*, but the heads were
#   defined as a single nn.Linear (classifier_X.weight). Keys didn't match,
#   so load_state_dict(strict=False) silently dropped all 12 tensors.
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import numpy as np
import json
import gc
import random
import time
from typing import Any
from dataclasses import dataclass
from enum import Enum

# ============================================================================
# SET SEED
# ============================================================================
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
print("🎲 Random seed set to 123")

# ============================================================================
# CONFIGURATION  (mirrors the reference notebook)
# ============================================================================
HF_REPO_ID    = 'frankmorales2020/topological-ai-mixtral-8x7b-multirun'
BASE_MODEL_ID = 'frankmorales2020/mixtral-8x7b-fp8-topo2026'
HIDDEN_SIZE   = 4096
HEAD_HIDDEN   = 512
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

TASK_CONFIG = {
    'A': {'name': 'World vs Sports',     'labels': {0: 'World', 1: 'Sports'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'B': {'name': 'Business vs Sci/Tech','labels': {0: 'Business', 1: 'Sci/Tech'},
          'confidence_threshold': 0.75, 'fallback_task': None, 'bias_correction': 0.0},
    'C': {'name': 'World vs Sci/Tech',   'labels': {0: 'World', 1: 'Sci/Tech'},
          'confidence_threshold': 0.80, 'fallback_task': 'B', 'bias_correction': -0.5},
}

# ============================================================================
# DISABLE COMPRESSED TENSORS HOOKS
# ============================================================================
def disable_compressed_tensors_hooks(model):
    for module in model.modules():
        for hook_dict_name in ('_forward_pre_hooks', '_forward_hooks'):
            hook_dict = getattr(module, hook_dict_name, None)
            if hook_dict:
                to_remove = [hid for hid, hook in hook_dict.items()
                             if 'compress' in str(hook).lower() or 'decompress' in str(hook).lower()]
                for hid in to_remove:
                    del hook_dict[hid]
    return model

# ============================================================================
# LOAD REAL MODEL
# ============================================================================
print("=" * 70)
print("LOADING REAL MIXTRAL MODEL")
print("=" * 70)

print(f"📥 Loading base model from {BASE_MODEL_ID}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map=None,
    low_cpu_mem_usage=False,
    ignore_mismatched_sizes=True,
)
base_model = base_model.cpu()

print("🔧 Removing compression hooks...")
base_model = disable_compressed_tensors_hooks(base_model)

print("🔄 Converting FP8 tensors to bfloat16...")
converted = 0
for _, param in base_model.named_parameters():
    if param.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        param.data = param.data.to(torch.bfloat16); converted += 1
for _, buffer in base_model.named_buffers():
    if buffer.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
        buffer.data = buffer.data.to(torch.bfloat16); converted += 1
print(f"   Converted {converted} FP8 tensors")

print("🧹 Removing compression attributes...")
attrs_removed = 0
for module in base_model.modules():
    for attr in ['input_scale', 'weight_scale', 'down_proj_scale', 'gate_up_proj_scale',
                 'scales', 'zero_points', 'quantization_scheme', 'compression_config']:
        if hasattr(module, attr):
            delattr(module, attr); attrs_removed += 1
print(f"   Removed {attrs_removed} compression attributes")

gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================================
# TASK-AWARE MODEL  (head arch matches the checkpoint: Sequential .0/.1/.2/.3)
# ============================================================================
class RealTaskAwareModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        for param in self.base_model.parameters():
            param.requires_grad = False

        def _head():
            return nn.Sequential(
                nn.Linear(HIDDEN_SIZE, HEAD_HIDDEN, dtype=torch.bfloat16),  # .0
                nn.GELU(),                                                  # .1
                nn.Dropout(0.1),                                            # .2
                nn.Linear(HEAD_HIDDEN, 2, dtype=torch.bfloat16),           # .3
            )

        self.classifier_A = _head()
        self.classifier_B = _head()
        self.classifier_C = _head()
        self.current_task = 'A'
        self.loaded_heads = set()   # populated after verified checkpoint load
        self._init_heads()

    def _init_heads(self):
        for head in (self.classifier_A, self.classifier_B, self.classifier_C):
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def switch_task(self, task):
        self.current_task = task

    def forward(self, input_ids, attention_mask=None):
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            hidden = outputs.hidden_states[-1]

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)

        classifier = getattr(self, f'classifier_{self.current_task}')
        return classifier(pooled)

    @torch.no_grad()
    def predict_task(self, text, task='A'):
        """Single-head prediction with the task's bias correction applied."""
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        self.switch_task(task)
        logits = self.forward(inputs['input_ids'], inputs['attention_mask'])

        bias = TASK_CONFIG[task].get('bias_correction', 0.0)
        if bias != 0.0:
            logits[:, 0] += bias

        probs = F.softmax(logits.float(), dim=-1)
        pred_class = probs.argmax().item()
        confidence = probs.max().item()
        label = TASK_CONFIG[task]['labels'][pred_class]
        return label, confidence, pred_class

    @torch.no_grad()
    def predict_with_fallback(self, text, task='C'):
        """Primary prediction; fall back to fallback_task if below threshold."""
        cfg = TASK_CONFIG[task]
        label, conf, cls = self.predict_task(text, task)
        if conf >= cfg['confidence_threshold'] or not cfg.get('fallback_task'):
            return label, conf, 'primary'
        fb = cfg['fallback_task']
        fb_label, fb_conf, _ = self.predict_task(text, fb)
        return fb_label, fb_conf, f'fallback->{fb}'

# ============================================================================
# CREATE MODEL + VERIFIED CHECKPOINT LOAD
# ============================================================================
print("\n" + "=" * 70)
print("CREATING TASK-AWARE MODEL")
print("=" * 70)

real_model = RealTaskAwareModel(base_model)

def head_signature(model, name):
    seq = getattr(model, name)
    sig = []
    for layer in seq:
        if isinstance(layer, nn.Linear):
            sig.append(round(layer.weight.float().abs().sum().item(), 4))
            sig.append(round(layer.bias.float().abs().sum().item(), 4))
    return tuple(sig)

pre_load_sig = {n: head_signature(real_model, n)
                for n in ('classifier_A', 'classifier_B', 'classifier_C')}

print("📦 Attempting to load certified weights...")
try:
    checkpoint_path = hf_hub_download(repo_id=HF_REPO_ID, filename='certified_topological_best.pt')
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Match checkpoint dtype to the bfloat16 heads.
    for k, v in checkpoint.items():
        if isinstance(v, torch.Tensor) and v.dtype == torch.float32:
            checkpoint[k] = v.to(torch.bfloat16)

    filtered = {k: v for k, v in checkpoint.items() if k.startswith('classifier_')}
    missing, unexpected = real_model.load_state_dict(filtered, strict=False)
    print(f"✓ Filtered {len(filtered)} classifier tensors from checkpoint")

    for name in ('classifier_A', 'classifier_B', 'classifier_C'):
        if head_signature(real_model, name) != pre_load_sig[name]:
            real_model.loaded_heads.add(name[-1])

    trusted = sorted(real_model.loaded_heads)
    untrusted = [t for t in ('A', 'B', 'C') if t not in real_model.loaded_heads]
    print(f"  ✅ Trained heads loaded: {trusted if trusted else 'NONE'}")
    if untrusted:
        print(f"  ⚠️  Heads still at RANDOM init: {untrusted}")
    cls_unexpected = [k for k in unexpected if 'classifier' in k]
    if cls_unexpected:
        print(f"  ⚠️  Unexpected classifier keys (arch mismatch): {cls_unexpected[:6]}")
except Exception as e:
    print(f"⚠️ Could not load certified weights: {e}")
    print("   Using randomly initialized classifiers — outputs are NOT meaningful.")

del pre_load_sig
print(f"📱 Moving model to {DEVICE}...")
real_model = real_model.to(DEVICE)
real_model.eval()
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print(f"✓ Base dtype: {next(real_model.base_model.parameters()).dtype}")
print(f"✓ Head dtype: {next(real_model.classifier_A.parameters()).dtype}")

# ============================================================================
# AGENTS
# ============================================================================
class AgentRole(Enum):
    CLASSIFIER = "classifier"
    TOPIC = "topic"
    SENTIMENT = "sentiment"
    DECISION = "decision"

@dataclass
class AgentOutput:
    agent: str
    result: Any
    confidence: float

class RealClassifierAgent:
    """
    The three heads are binary classifiers on different axes, so voting across
    them is invalid (Task B can't express "Sports", etc.). Instead we ROUTE each
    document to the single head whose axis fits, using keyword signals, then
    trust only that head. Task C keeps its bias-correction + fallback-to-B.
    """
    # Keyword signals -> which binary head can actually decide this document.
    ROUTING = {
        'A': ['game', 'team', 'win', 'won', 'championship', 'league', 'match',
              'victory', 'series', 'cup', 'tournament', 'season', 'player',
              'coach', 'score', 'playoff'],
        'B': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding',
              'profit', 'shares', 'quarterly', 'investor', 'ipo', 'merger'],
    }

    def __init__(self, model):
        self.model = model

    # Health/science/general-news terms belong on the World axis; if these are
    # present we route to C and suppress the B fallback (B can't express World).
    WORLD_SIGNAL = ['cancer', 'treatment', 'medical', 'drug', 'disease', 'patient',
                    'vaccine', 'health', 'president', 'government', 'election',
                    'war', 'climate', 'court', 'nasa', 'space', 'mission']

    def _route(self, text):
        tl = text.lower()
        a_hits = sum(1 for kw in self.ROUTING['A'] if kw in tl)
        b_hits = sum(1 for kw in self.ROUTING['B'] if kw in tl)
        w_hits = sum(1 for kw in self.WORLD_SIGNAL if kw in tl)
        if a_hits and a_hits >= b_hits:
            return 'A'
        # Strong world/health/science signal outweighs a stray business keyword.
        if w_hits and w_hits >= b_hits:
            return 'C'
        if b_hits:
            return 'B'
        return 'C'

    def process(self, text):
        task = self._route(text)

        if task == 'C':
            tl = text.lower()
            has_world = any(kw in tl for kw in self.WORLD_SIGNAL)
            if has_world:
                # Trust C directly; B's fallback can't express World and would
                # mislabel general/health news as Business/Sci-Tech.
                label, conf, _ = self.model.predict_task(text, 'C')
                source = 'primary'
            else:
                label, conf, source = self.model.predict_with_fallback(text, 'C')
        else:
            label, conf, _ = self.model.predict_task(text, task)
            source = 'primary'

        return AgentOutput(
            agent='Classifier',
            result={
                'category': label,
                'routed_task': task,
                'task_c_source': source,
            },
            confidence=conf
        )

class RealTopicAgent:
    def process(self, text):
        topics = []
        text_lower = text.lower()
        topic_keywords = {
            'sports': ['game', 'team', 'win', 'championship', 'league', 'match', 'victory'],
            'business': ['stock', 'earnings', 'revenue', 'sales', 'market', 'funding', 'profit'],
            'technology': ['ai', 'quantum', 'software', 'cloud', 'algorithm', 'chip', 'computing'],
            'health': ['cancer', 'treatment', 'medical', 'drug', 'disease', 'breakthrough'],
            'politics': ['president', 'government', 'policy', 'election', 'law', 'congress'],
            'space': ['nasa', 'mission', 'space', 'moon', 'mars', 'jupiter', 'launch'],
        }
        for topic, keywords in topic_keywords.items():
            if any(kw in text_lower for kw in keywords):
                topics.append(topic)
        return AgentOutput(agent='Topic', result=topics if topics else ['general'],
                           confidence=0.8 if topics else 0.5)

class RealSentimentAgent:
    def process(self, text):
        text_lower = text.lower()
        positive = ['won', 'victory', 'breakthrough', 'growth', 'strong', 'record', 'beat', 'secured', 'surged']
        negative = ['loss', 'defeat', 'crisis', 'decline', 'risk', 'concern', 'failed']
        pos = sum(1 for w in positive if w in text_lower)
        neg = sum(1 for w in negative if w in text_lower)
        if pos > neg:
            return AgentOutput(agent='Sentiment', result='positive', confidence=0.8)
        elif neg > pos:
            return AgentOutput(agent='Sentiment', result='negative', confidence=0.8)
        return AgentOutput(agent='Sentiment', result='neutral', confidence=0.5)

class RealDecisionAgent:
    def process(self, classifier_out, topic_out, sentiment_out):
        category = classifier_out.result['category']
        confidence = classifier_out.confidence
        source = classifier_out.result.get('task_c_source', 'primary')
        is_fallback = source.startswith('fallback')
        # Auto-approve only confident, non-fallback predictions.
        if confidence >= 0.75 and not is_fallback:
            action, review = 'auto_approve', False
        else:
            action, review = 'review', True
        routing = {
            'Sports': 'sports_desk', 'Business': 'finance_team',
            'Sci/Tech': 'tech_research', 'World': 'news_department',
        }.get(category, 'general_inbox')
        return AgentOutput(
            agent='Decision',
            result={'category': category, 'action': action, 'routing': routing,
                    'requires_review': review, 'topics': topic_out.result,
                    'sentiment': sentiment_out.result},
            confidence=confidence
        )

# ============================================================================
# ORCHESTRATOR
# ============================================================================
class RealOrchestrator:
    def __init__(self, model):
        self.classifier = RealClassifierAgent(model)
        self.topic = RealTopicAgent()
        self.sentiment = RealSentimentAgent()
        self.decision = RealDecisionAgent()

    def analyze(self, text, verbose=True):
        start = time.time()
        classifier_out = self.classifier.process(text)
        topic_out = self.topic.process(text)
        sentiment_out = self.sentiment.process(text)
        decision_out = self.decision.process(classifier_out, topic_out, sentiment_out)
        elapsed = (time.time() - start) * 1000

        if verbose:
            c = classifier_out.result
            print(f"\n{'=' * 55}")
            print(f"📝 {text[:65]}...")
            print(f"{'=' * 55}")
            print(f"🎯 {decision_out.result['category']} ({classifier_out.confidence * 100:.0f}%)"
                  f"  [routed->Task {c['routed_task']}, source {c['task_c_source']}]")
            print(f"🏷️  {', '.join(topic_out.result[:3])}")
            print(f"💭 {sentiment_out.result}")
            print(f"✅ {decision_out.result['action']} → {decision_out.result['routing']}")
            print(f"⏱️  {elapsed:.0f}ms")

        return {
            'text': text[:100],
            'category': decision_out.result['category'],
            'confidence': classifier_out.confidence,
            'routed_task': classifier_out.result['routed_task'],
            'task_c_source': classifier_out.result['task_c_source'],
            'topics': topic_out.result,
            'sentiment': sentiment_out.result,
            'action': decision_out.result['action'],
            'routing': decision_out.result['routing'],
            'time_ms': elapsed,
        }

# ============================================================================
# RUN
# ============================================================================
print("\n" + "=" * 70)
print("RUNNING REAL MULTI-AGENT SYSTEM")
print("=" * 70)

orchestrator = RealOrchestrator(real_model)

test_docs = [
    "The Chicago Cubs won the World Series championship",
    "Microsoft stock surged 15% after beating earnings",
    "NASA launched a mission to explore Jupiter's moon Europa",
    "Scientists discovered breakthrough treatment for cancer",
    "Apple reported record iPhone sales and new AI features",
]

results = []
for doc in test_docs:
    results.append(orchestrator.analyze(doc, verbose=True))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
for i, r in enumerate(results, 1):
    icon = "✅" if r['action'] == 'auto_approve' else "⚠️"
    print(f"{icon} {i}. {r['category']:10s} | {r['action']:12s} | "
          f"{r['confidence'] * 100:.0f}% (Task {r['routed_task']})")

with open('real_model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n💾 Saved to 'real_model_results.json'")
print("\n✅ REAL MODEL SYSTEM READY")

# ============================================================================
# NAN DETECTION - ADDED AFTER YOUR CODE (NO MODIFICATIONS TO YOUR CODE ABOVE)
# ============================================================================
print("\n" + "=" * 70)
print("🔍 NAN DETECTION")
print("=" * 70)

nan_results = {"has_nan": False, "has_inf": False}

# Check embedding layer
emb = real_model.base_model.get_input_embeddings()
weight = emb.weight.float()

nan_count = torch.isnan(weight).sum().item()
inf_count = torch.isinf(weight).sum().item()

print(f"\n📊 Embedding Matrix ({weight.numel():,} elements):")
print(f"   Shape: {tuple(weight.shape)}")
print(f"   Dtype: {emb.weight.dtype}")
print(f"   NaN: {nan_count:,}")
print(f"   Inf: {inf_count:,}")
print(f"   Mean: {weight.mean().item():.6f}")
print(f"   Std: {weight.std().item():.6f}")
print(f"   Min: {weight.min().item():.6f}")
print(f"   Max: {weight.max().item():.6f}")

if nan_count > 0:
    print(f"   ❌ CONTAINS {nan_count:,} NaN VALUES!")
    nan_results["has_nan"] = True
else:
    print(f"   ✅ No NaN values")

if inf_count > 0:
    print(f"   ❌ CONTAINS {inf_count:,} Inf VALUES!")
    nan_results["has_inf"] = True
else:
    print(f"   ✅ No Inf values")

# Check prime anchor rows
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
print(f"\n🎯 Prime Anchors: {PRIME_ANCHORS}")
anchor_nan = 0
anchor_inf = 0

for idx in PRIME_ANCHORS:
    if idx < weight.shape[0]:
        row = weight[idx]
        a_nan = torch.isnan(row).sum().item()
        a_inf = torch.isinf(row).sum().item()
        anchor_nan += a_nan
        anchor_inf += a_inf
        status = "✅" if a_nan == 0 and a_inf == 0 else "❌"
        print(f"   {status} Anchor {idx}: NaN={a_nan}, Inf={a_inf}")

if anchor_nan > 0:
    nan_results["has_nan"] = True
    print(f"   ❌ Anchor rows contain {anchor_nan} NaN values")
else:
    print(f"   ✅ No NaN in anchor rows")

# Check classifier heads
print(f"\n📊 Classifier Heads:")
for head_name in ['classifier_A', 'classifier_B', 'classifier_C']:
    head = getattr(real_model, head_name)
    head_nan = 0
    head_inf = 0
    total_params = 0

    for param in head.parameters():
        p_f32 = param.float()
        head_nan += torch.isnan(p_f32).sum().item()
        head_inf += torch.isinf(p_f32).sum().item()
        total_params += p_f32.numel()

    status = "✅" if head_nan == 0 and head_inf == 0 else "❌"
    print(f"   {status} {head_name}: params={total_params:,}, NaN={head_nan}, Inf={head_inf}")

    if head_nan > 0 or head_inf > 0:
        nan_results["has_nan"] = True
        nan_results["has_inf"] = True

# Final summary
status = "PASS" if not nan_results["has_nan"] and not nan_results["has_inf"] else "FAIL"
print("\n" + "=" * 80)
print("📋 NAN DETECTION SUMMARY")
print("=" * 80)
print(f"   Status: {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
print(f"   Embedding NaN: {'❌ YES' if nan_results['has_nan'] else '✅ NO'}")
print(f"   Embedding Inf: {'❌ YES' if nan_results['has_inf'] else '✅ NO'}")
print(f"   Anchor NaN: {'❌ YES' if anchor_nan > 0 else '✅ NO'}")
print("=" * 80)
print("\n✅ Mixtral NaN check complete")

🎲 Random seed set to 123
LOADING REAL MIXTRAL MODEL
📥 Loading base model from frankmorales2020/mixtral-8x7b-fp8-topo2026...


Compressing model: 100%|██████████| 128/128 [00:00<00:00, 1969.82it/s]


Loading weights:   0%|          | 0/547 [00:00<?, ?it/s]

[transformers] MixtralForCausalLM LOAD REPORT from: frankmorales2020/mixtral-8x7b-fp8-topo2026
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w2.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w3.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.down_proj_scale                         | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.{0, 1, 2, 3, 4, 5, 6, 7}.w1.input_scale | UNEXPECTED |  | 
model.layers.{0...31}.mlp.experts.gate_up_proj_scale                      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔧 Removing compression hooks...
🔄 Converting FP8 tensors to bfloat16...
   Converted 192 FP8 tensors
🧹 Removing compression attributes...
   Removed 384 compression attributes
📚 Loading tokenizer...

CREATING TASK-AWARE MODEL
📦 Attempting to load certified weights...
✓ Filtered 12 classifier tensors from checkpoint
  ✅ Trained heads loaded: ['A', 'B', 'C']
📱 Moving model to cuda...
✓ Base dtype: torch.bfloat16
✓ Head dtype: torch.bfloat16

RUNNING REAL MULTI-AGENT SYSTEM

📝 The Chicago Cubs won the World Series championship...
🎯 Sports (92%)  [routed->Task A, source primary]
🏷️  sports
💭 positive
✅ auto_approve → sports_desk
⏱️  1130ms

📝 Microsoft stock surged 15% after beating earnings...
🎯 Business (98%)  [routed->Task B, source primary]
🏷️  business
💭 positive
✅ auto_approve → finance_team
⏱️  67ms

📝 NASA launched a mission to explore Jupiter's moon Europa...
🎯 World (94%)  [routed->Task C, source primary]
🏷️  space
💭 neutral
✅ auto_approve → news_department
⏱️  67ms

📝 Scientists

## GPT-OSS

GPT0SS20B_TOPOAI_TRANSFORMER_MULTRUN_CORRECTED.ipynb

In [ ]:
!nvidia-smi

Sun Jun 21 12:38:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# ============================================================================
# 11. STANDALONE HF INFERENCE — run independently, no prior cells needed
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, math
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download

HF_REPO_ID    = 'frankmorales2020/topological-ai-gpt-oss-20b-multirun'
BASE_MODEL_ID = 'openai/gpt-oss-20b'
HIDDEN_SIZE   = 2880
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TASK_LABELS   = {'A':{0:'World',1:'Sports'},'B':{0:'Business',1:'Sci/Tech'},'C':{0:'World',1:'Sci/Tech'}}
TEST_INPUTS   = [
    ('A', 'The national team won the championship after a stunning comeback victory.'),
    ('B', 'Quarterly earnings beat analyst expectations driven by strong cloud revenue growth.'),
    ('C', 'New quantum computing startup secures massive initial funding round for enterprise deployment.'),
]

class TaskAwareModel(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base_model   = base
        dev = next(base.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'
    def forward(self, input_ids, attention_mask=None):
        h = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1]
        if attention_mask is not None:
            h = h[torch.arange(input_ids.shape[0], device=input_ids.device), torch.eq(attention_mask,1).int().sum(-1)-1, :]
        else:
            h = h[:,-1,:]
        return getattr(self, f'classifier_{self.current_task}')(h)
    def switch_task(self, t): self.current_task = t

print('='*75)
print(f'TOPO-2026 HF INFERENCE  |  {HF_REPO_ID}')
print(f'Safety Constant Λ: {1.0 - math.prod(1.0-p**-0.5 for p in [2,3,5,7,11,13]):.10f}')
print('='*75)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device)
for p in base.parameters(): p.requires_grad = False
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tok.pad_token = tok.eos_token
m = TaskAwareModel(base)
m.load_state_dict(torch.load(hf_hub_download(repo_id=HF_REPO_ID, filename='certified_topological_best.pt'), map_location='cpu'), strict=False)
m.eval()
print('✓ Certified checkpoint loaded from Hub\n')
for task, sentence in TEST_INPUTS:
    inp = tok(sentence, return_tensors='pt', max_length=64, padding='max_length', truncation=True).to(device)
    with torch.no_grad():
        m.switch_task(task)
        probs = F.softmax(m(inp.input_ids, inp.attention_mask).float(), dim=-1).squeeze().cpu().numpy()
    idx, conf = int(np.argmax(probs)), float(probs.max())
    status = '✓ CERTIFIED' if conf >= 0.85 else '~ PASS' if conf >= 0.70 else '✗ LOW'
    print(f'Task {task} [{status}]  {TASK_LABELS[task][idx]:10s}  {conf*100:.2f}%  "{sentence[:65]}"')
print('\n'+'='*75)

# ============================================================================
# NAN DETECTION - ADDED AFTER YOUR CODE (NO MODIFICATIONS TO YOUR CODE ABOVE)
# ============================================================================
print("\n" + "=" * 75)
print("🔍 NAN DETECTION")
print("=" * 75)

nan_results = {"has_nan": False, "has_inf": False}

# Check embedding layer
emb = m.base_model.get_input_embeddings()
weight = emb.weight.float()

nan_count = torch.isnan(weight).sum().item()
inf_count = torch.isinf(weight).sum().item()

print(f"\n📊 Embedding Matrix ({weight.numel():,} elements):")
print(f"   Shape: {tuple(weight.shape)}")
print(f"   Dtype: {emb.weight.dtype}")
print(f"   NaN: {nan_count:,}")
print(f"   Inf: {inf_count:,}")
print(f"   Mean: {weight.mean().item():.6f}")
print(f"   Std: {weight.std().item():.6f}")
print(f"   Min: {weight.min().item():.6f}")
print(f"   Max: {weight.max().item():.6f}")

if nan_count > 0:
    print(f"   ❌ CONTAINS {nan_count:,} NaN VALUES!")
    nan_results["has_nan"] = True
else:
    print(f"   ✅ No NaN values")

if inf_count > 0:
    print(f"   ❌ CONTAINS {inf_count:,} Inf VALUES!")
    nan_results["has_inf"] = True
else:
    print(f"   ✅ No Inf values")

# Check prime anchor rows
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
print(f"\n🎯 Prime Anchors: {PRIME_ANCHORS}")
anchor_nan = 0
anchor_inf = 0

for idx in PRIME_ANCHORS:
    if idx < weight.shape[0]:
        row = weight[idx]
        a_nan = torch.isnan(row).sum().item()
        a_inf = torch.isinf(row).sum().item()
        anchor_nan += a_nan
        anchor_inf += a_inf
        status = "✅" if a_nan == 0 and a_inf == 0 else "❌"
        print(f"   {status} Anchor {idx}: NaN={a_nan}, Inf={a_inf}")

if anchor_nan > 0:
    nan_results["has_nan"] = True
    print(f"   ❌ Anchor rows contain {anchor_nan} NaN values")
else:
    print(f"   ✅ No NaN in anchor rows")

# Check classifier heads (Linear layers)
print(f"\n📊 Classifier Heads:")
for head_name in ['classifier_A', 'classifier_B', 'classifier_C']:
    head = getattr(m, head_name)
    head_nan = 0
    head_inf = 0
    total_params = 0

    for param in head.parameters():
        p_f32 = param.float()
        head_nan += torch.isnan(p_f32).sum().item()
        head_inf += torch.isinf(p_f32).sum().item()
        total_params += p_f32.numel()

    status = "✅" if head_nan == 0 and head_inf == 0 else "❌"
    print(f"   {status} {head_name}: params={total_params:,}, NaN={head_nan}, Inf={head_inf}")

    if head_nan > 0 or head_inf > 0:
        nan_results["has_nan"] = True
        nan_results["has_inf"] = True

# Check LM head
if hasattr(m.base_model, 'lm_head'):
    print(f"\n📊 LM Head:")
    lm_weight = m.base_model.lm_head.weight.float()
    lm_nan = torch.isnan(lm_weight).sum().item()
    lm_inf = torch.isinf(lm_weight).sum().item()
    status = "✅" if lm_nan == 0 and lm_inf == 0 else "❌"
    print(f"   {status} LM Head: shape={tuple(lm_weight.shape)}, NaN={lm_nan}, Inf={lm_inf}")
    if lm_nan > 0 or lm_inf > 0:
        nan_results["has_nan"] = True
        nan_results["has_inf"] = True

# Final summary
status = "PASS" if not nan_results["has_nan"] and not nan_results["has_inf"] else "FAIL"
print("\n" + "=" * 75)
print("📋 NAN DETECTION SUMMARY")
print("=" * 75)
print(f"   Status: {'✅ PASS' if status == 'PASS' else '❌ FAIL'}")
print(f"   Embedding NaN: {'❌ YES' if nan_results['has_nan'] else '✅ NO'}")
print(f"   Embedding Inf: {'❌ YES' if nan_results['has_inf'] else '✅ NO'}")
print(f"   Anchor NaN: {'❌ YES' if anchor_nan > 0 else '✅ NO'}")
print("=" * 75)
print("\n✅ GPT-OSS-20B NaN check complete")

TOPO-2026 HF INFERENCE  |  frankmorales2020/topological-ai-gpt-oss-20b-multirun
Safety Constant Λ: 0.9785142874


config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


model.safetensors.index.json:   0%|          | 0.00/36.4k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

certified_topological_best.pt:   0%|          | 0.00/41.8G [00:00<?, ?B/s]

✓ Certified checkpoint loaded from Hub

Task A [✓ CERTIFIED]  Sports      100.00%  "The national team won the championship after a stunning comeback "
Task B [✓ CERTIFIED]  Sci/Tech    100.00%  "Quarterly earnings beat analyst expectations driven by strong clo"
Task C [✓ CERTIFIED]  Sci/Tech    100.00%  "New quantum computing startup secures massive initial funding rou"


🔍 NAN DETECTION

📊 Embedding Matrix (579,133,440 elements):
   Shape: (201088, 2880)
   Dtype: torch.bfloat16
   NaN: 0
   Inf: 0
   Mean: 0.000590
   Std: 2.405696
   Min: -104.500000
   Max: 118.500000
   ✅ No NaN values
   ✅ No Inf values

🎯 Prime Anchors: [2, 3, 5, 7, 11, 13]
   ✅ Anchor 2: NaN=0, Inf=0
   ✅ Anchor 3: NaN=0, Inf=0
   ✅ Anchor 5: NaN=0, Inf=0
   ✅ Anchor 7: NaN=0, Inf=0
   ✅ Anchor 11: NaN=0, Inf=0
   ✅ Anchor 13: NaN=0, Inf=0
   ✅ No NaN in anchor rows

📊 Classifier Heads:
   ✅ classifier_A: params=5,762, NaN=0, Inf=0
   ✅ classifier_B: params=5,762, NaN=0, Inf=0
   ✅ classifier_C: params=5,762, N